# Module 04 — Function Approximation

**Reinforcement Learning · Dakar Institute of Technology (DIT)**

Tabular methods store one value per state. That collapses when the state space is
**large or continuous** — you can't have a table row for every real-valued
position and velocity. The fix: **approximate** the value function with a
parameterized function $\hat q(s,a;\mathbf{w})$ and learn the parameters $\mathbf{w}$
by (semi-)gradient descent.

We cover two levels:
1. **Linear** function approximation with **tile coding** features →
   *semi-gradient SARSA* on **MountainCar-v0**.
2. **Deep** function approximation → a **DQN** (Deep Q-Network) on **CartPole-v1**,
   with the two tricks that make it stable: **experience replay** and a
   **target network**.

### The semi-gradient update
$$\mathbf{w} \leftarrow \mathbf{w} + \alpha\big[r + \gamma\,\hat q(s',a';\mathbf{w}) - \hat q(s,a;\mathbf{w})\big]\,\nabla_{\mathbf{w}}\hat q(s,a;\mathbf{w})$$
It's "semi"-gradient because we treat the bootstrap target as a constant.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "rl_helpers").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from rl_helpers import set_seed, plot_learning_curve, animate_policy

set_seed(0)
rng = np.random.default_rng(0)
print("gymnasium:", gym.__version__)

## Part A — Linear FA with tile coding on MountainCar

**MountainCar-v0**: an underpowered car must build momentum to escape a valley.
State = `(position, velocity)` (continuous); 3 actions (push left / none / right).
Reward is **−1 every step** until the goal, so the agent must learn to reach the
flag in as few steps as possible.

**Tile coding** turns continuous coordinates into sparse binary features: we lay
several offset grids ("tilings") over the 2-D state space; each state activates one
tile per tiling. The value is a linear function of these binary features, which
lets a linear model represent non-linear value surfaces.

In [ ]:
env = gym.make("MountainCar-v0")
print("observation space:", env.observation_space) # Box: pos in [-1.2, 0.6], vel in [-0.07, 0.07]
print("action space: ", env.action_space) # Discrete(3)
low, high = env.observation_space.low, env.observation_space.high
print("low :", low)
print("high:", high)

In [ ]:
class TileCoder:
    '''Minimal grid tile coder for a 2-D continuous state.'''
    def __init__(self, low, high, n_tilings=8, n_bins=8):
        self.low = np.asarray(low, dtype=float)
        self.high = np.asarray(high, dtype=float)
        self.n_tilings = n_tilings
        self.n_bins = n_bins
        self.dim = len(low)
        self.tile_width = (self.high - self.low) / n_bins
        # each tiling is shifted by a fraction of a tile in every dimension
        self.offsets = [self.tile_width * (i / n_tilings) for i in range(n_tilings)]
        self.n_features = n_tilings * (n_bins + 1) ** self.dim

    def features(self, state):
        '''Return indices of the active tiles (one per tiling).'''
        state = np.asarray(state, dtype=float)
        active = []
        for t in range(self.n_tilings):
            coords = ((state - self.low + self.offsets[t]) / self.tile_width).astype(int)
            coords = np.clip(coords, 0, self.n_bins)
            flat = coords[0] * (self.n_bins + 1) + coords[1]
            active.append(t * (self.n_bins + 1) ** self.dim + flat)
        return np.array(active, dtype=int)

tc = TileCoder(low, high, n_tilings=8, n_bins=8)
print("total features:", tc.n_features, "| active per state:", tc.n_tilings)
print("example active tiles:", tc.features(env.reset(seed=0)[0]))

### Semi-gradient SARSA with linear features

Because tile features are binary (0/1), the value $\hat q(s,a)=\sum_{i\in\text{active}} w_{a,i}$
is just a **sum of the active weights**, and the gradient w.r.t. those weights is 1.
So the update simply nudges the active weights by $\alpha\cdot\delta$.

In [ ]:
def q_value(w, active, a):
    return w[a, active].sum()

def semigradient_sarsa(env, tc, n_episodes=400, alpha=0.1, gamma=1.0,
                       eps=0.0, seed=0):
    alpha = alpha / tc.n_tilings
    nA = env.action_space.n
    w = np.zeros((nA, tc.n_features))
    ep_returns = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=seed + ep)
        active = tc.features(s)
        a = int(np.argmax([q_value(w, active, b) for b in range(nA)]))
        G, done = 0.0, False
        while not done:
            ns, r, term, trunc, _ = env.step(a)
            done = term or trunc
            G += r
            q_sa = q_value(w, active, a)
            if done:
                # TODO: terminal update — target is just r (no next state)
                raise NotImplementedError("Terminal semi-gradient update")
                break
            nactive = tc.features(ns)
            na = int(np.argmax([q_value(w, nactive, b) for b in range(nA)]))
            # TODO: SARSA target = r + gamma * q(ns, na); update the ACTIVE weights
            # of (s, a): w[a, active] += alpha * (target - q_sa)
            raise NotImplementedError("Semi-gradient SARSA update")
            active, a = nactive, na
        ep_returns.append(G)
    return w, ep_returns

w, mc_returns = semigradient_sarsa(env, tc, n_episodes=400, alpha=0.3)
plot_learning_curve(mc_returns, window=20,
                    title="MountainCar: semi-gradient SARSA (return = -steps)")
plt.show()
print(f"Best episode return: {max(mc_returns):.0f} (closer to 0 is better)")

### Visualise the learned value surface & watch the car climb
The cost-to-go $-\max_a \hat q(s,a)$ over the 2-D state space is the classic
MountainCar "valley" plot. Then we render the greedy policy.

In [ ]:
pos = np.linspace(low[0], high[0], 60)
vel = np.linspace(low[1], high[1], 60)
cost = np.zeros((len(pos), len(vel)))
for i, p in enumerate(pos):
    for j, v in enumerate(vel):
        act = tc.features((p, v))
        cost[i, j] = -max(q_value(w, act, b) for b in range(env.action_space.n))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.contourf(vel, pos, cost, levels=20, cmap="viridis")
ax.set_xlabel("velocity"); ax.set_ylabel("position")
ax.set_title("Cost-to-go -max_a q(s,a)"); plt.colorbar(im, ax=ax)
plt.show()

def greedy(s):
    act = tc.features(s)
    return int(np.argmax([q_value(w, act, b) for b in range(env.action_space.n)]))

render_env = gym.make("MountainCar-v0", render_mode="rgb_array")
animate_policy(render_env, greedy, max_steps=250, fps=30, path="mountaincar.gif")

## Part B — Deep Q-Network (DQN) on CartPole

When we can't hand-design good features, let a **neural network** learn them.
**DQN** approximates $\hat q(s,a;\mathbf{w})$ with a network and stabilises training
with two ideas:

- **Experience replay:** store transitions in a buffer and train on random
  minibatches → breaks correlations between consecutive samples.
- **Target network:** a periodically-frozen copy of the network provides the
  bootstrap target → stops the target from chasing a moving prediction.

Loss (Huber/MSE) on a minibatch:
$$L = \big(r + \gamma\max_{a'}\hat q(s',a';\mathbf{w}^-) - \hat q(s,a;\mathbf{w})\big)^2$$
where $\mathbf{w}^-$ are the target-network weights.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import random

torch.manual_seed(0); random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

class QNet(nn.Module):
    def __init__(self, n_obs, n_act, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_act),
        )
    def forward(self, x):
        return self.net(x)

class ReplayBuffer:
    def __init__(self, capacity=50_000):
        self.buf = deque(maxlen=capacity)
    def push(self, *transition):
        self.buf.append(transition)
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (torch.tensor(np.array(s), dtype=torch.float32, device=device),
                torch.tensor(a, dtype=torch.int64, device=device).unsqueeze(1),
                torch.tensor(r, dtype=torch.float32, device=device).unsqueeze(1),
                torch.tensor(np.array(ns), dtype=torch.float32, device=device),
                torch.tensor(d, dtype=torch.float32, device=device).unsqueeze(1))
    def __len__(self):
        return len(self.buf)

In [ ]:
def train_dqn(n_episodes=250, gamma=0.99, lr=1e-3, batch_size=64,
              target_update=10, eps_start=1.0, eps_end=0.05, eps_decay=0.995):
    env = gym.make("CartPole-v1")
    n_obs = env.observation_space.shape[0]
    n_act = env.action_space.n
    q_net = QNet(n_obs, n_act).to(device)
    target_net = QNet(n_obs, n_act).to(device)
    target_net.load_state_dict(q_net.state_dict())
    opt = torch.optim.Adam(q_net.parameters(), lr=lr)
    buffer = ReplayBuffer()
    eps = eps_start
    ep_returns = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep)
        G, done = 0.0, False
        while not done:
            if random.random() < eps:
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    qs = q_net(torch.tensor(s, dtype=torch.float32, device=device))
                    a = int(qs.argmax().item())
            ns, r, term, trunc, _ = env.step(a)
            done = term or trunc
            buffer.push(s, a, r, ns, float(done))
            s = ns; G += r
            if len(buffer) >= batch_size:
                bs, ba, br, bns, bd = buffer.sample(batch_size)
                q_sa = q_net(bs).gather(1, ba) # Q(s,a) for taken actions
                with torch.no_grad():
                    # TODO: bootstrap target with the TARGET network:
                    # max_q_ns = target_net(bns).max(1, keepdim=True)[0]
                    # target = br + gamma * max_q_ns * (1 - bd)
                    raise NotImplementedError("Compute the DQN target")
                # TODO: loss = smooth_l1_loss(q_sa, target); backprop; opt.step()
                raise NotImplementedError("Optimise the Q-network")
        eps = max(eps_end, eps * eps_decay)
        if ep % target_update == 0:
            target_net.load_state_dict(q_net.state_dict()) # sync target network
        ep_returns.append(G)
    env.close()
    return q_net, ep_returns

q_net, dqn_returns = train_dqn(n_episodes=250)
plot_learning_curve(dqn_returns, window=20, title="DQN on CartPole-v1")
plt.show()

### Watch the trained DQN balance the pole

In [ ]:
render_env = gym.make("CartPole-v1", render_mode="rgb_array")
def dqn_policy(s):
    with torch.no_grad():
        return int(q_net(torch.tensor(s, dtype=torch.float32, device=device)).argmax())
animate_policy(render_env, dqn_policy, max_steps=500, fps=30, path="dqn_cartpole.gif")

## Recap
- **Function approximation** replaces the value table with a parameterized
  $\hat q(s,a;\mathbf{w})$ trained by **semi-gradient** updates.
- **Tile coding** + linear weights is a cheap, robust way to handle low-dim
  continuous states (MountainCar).
- **DQN** scales this to neural networks with **experience replay** + a
  **target network** for stability (CartPole).

### Going further (optional, same environments)
- Tune the tile-coding resolution (`n_tilings`, `n_bins`) and step size on
  MountainCar; how few episodes can you solve it in?
- Ablate DQN on CartPole: remove the target network (or the replay buffer) and
  watch training destabilise. Why does each trick help?

**Next:** Module 05 — model-based RL: *learn* a model and plan with it (Dyna-Q).